# AQDrop Job Submission and Retrieval Example

This notebook demonstrates how to submit a simple Bell-state quantum circuit to an AQDrop queue and then retrieve the results.

In [7]:
import os
from qiskit import QuantumCircuit
from AqdropUser import AqdropUser

# --- Configuration ---
QUEUE_NAME = "X6Y3"  # Replace with your actual queue name
SHOTS = 4000
VERBOSITY = 3

In [8]:
def circ_bell():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure(0, 0)
    qc.measure(1, 1)
    return qc

qc = circ_bell()
print("Bell State Circuit:")
print(qc.draw())

Bell State Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [9]:
# 1) Instantiate user
user = AqdropUser(VERBOSITY)

# 2) Assemble job input
circuits = [qc]
job_meta = {
    "shots": [SHOTS], 
    "comment": "Notebook Bell State Job", 
    "queue_name": QUEUE_NAME, 
    "pref_qubits": None
}
user.assemble_job_input(circuits, job_meta)

# 3) Push to DB and get job ID
job_id = user.push_job_input()
print(f"Job submitted successfully! Job ID: {job_id}")

assembled job_input for 1 circuits, queue=X6Y3
idx  num_qubits  num_shots
  0           2  4000
pushed job_input; assigned job_id=1082
   ./job_retrieve.py --id 1082
Job submitted successfully! Job ID: 1082


## Retrieving the Job

Now we will retrieve the results for the job we just submitted. Note that depending on the queue, the job might still be `queued`.

In [10]:
# Retrieve the job using the ID from the previous step
job = user.pull_job(job_id)
print(f"Job Status: {job['status']}")

pulled job: {'id': 1082, 'owner_name': 'evan_u', 'queue_name': 'X6Y3', 'status': 'queued'}
Job Status: queued


In [11]:
# Parse the job into its components
circL, inputMD, output, transpiledL = user.parse_job()

if job['status'] == "success":
    user.print_shot_summary()
    print(f"Total Execution Time: {output['tot_exec_time']:.1f} sec")
    print(f"Received Total Shots: {output['tot_shots']}")
    print(f"Calibration Version: {output['calib_ver']}")
    print(f"Execution Date: {output['exec_date']}")
    
    if VERBOSITY > 0:
        user.print_output_counts()
else:
    print("Results are not available yet or the job failed.")

parsed job: 1 input circuits, no output yet (status=queued)
Results are not available yet or the job failed.


In [6]:
# Display circuits and transpiled circuits
print(f"Packed circuits: {len(circL)}, total requested shots: {sum(inputMD['shots'])}")

if VERBOSITY > 1:
    for idx, qc in enumerate(circL):
        print(f"\nCircuit {idx}:")
        print(qc)
    if transpiledL:
        for idx, qc in enumerate(transpiledL):
            print(f"\nTranspiled Circuit {idx}:")
            print(qc)

Packed circuits: 1, total requested shots: 4000
